In [ ]:
# ! pip install -q torch transformers datasets evaluate accelerate jiwer
# ! pip uninstall -y torchcodec
# ! sudo apt-get update -y
# ! sudo apt-get install -y ffmpeg
# ! pip install -q "torchcodec==0.7.*" --index-url https://download.pytorch.org/whl/cu126

Automatic speech recognition (ASR) converts a speech signal to text, mapping a sequence of audio inputs to text outputs.

## Load dependencies

In [ ]:
from functools import partial
import evaluate
import torch
import numpy as np
from huggingface_hub import notebook_login
from datasets import load_dataset, Audio
from transformers import AutoProcessor, pipeline
from transformers import AutoModelForCTC, TrainingArguments, Trainer
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

## Config

In [ ]:
user_name = "amin-oj"
dataset_dict = {"path": "PolyAI/minds14", "name": "en-US", "split": "train[:100]"}
# model_name = "facebook/wav2vec2-base"
    # This model needs a lot of training to output reasonable results
    # because it has been trained on audio only, and no prior SFT is applied to it
# model_name = "Thienpkae/wav2vec2-vivos-asr"
    # This is a community finetuned model. still needs heavy SFT
model_name = "facebook/wav2vec2-base-960h" # This is already perfect, but the loss trend is meaningful
push_to_hub=True
train_bs = 8
eval_bs = 8
max_train_steps = 500
num_train_epochs = 1
task = "asr"
ckpt_name = f"{model_name.split('/')[-1]}-finetuned-{task}-{dataset_dict['path'].replace("/", "_")}-{dataset_dict['name']}"

## HF Login

In [ ]:
notebook_login()

## Load dataset

In [ ]:
minds = load_dataset(**dataset_dict)
minds = minds.train_test_split(test_size=0.2)
print(minds["train"].features.keys())
minds = minds.remove_columns(["english_transcription", "intent_class", "lang_id"])
print(minds["train"].features)

There are two fields:

- `audio`: a 1-dimensional `array` of the speech signal that must be called to load and resample the audio file.
- `transcription`: the target text.

## Preprocess

- The MInDS-14 dataset has a sampling rate of 8000Hz (you can find this information in its [dataset card](https://huggingface.co/datasets/PolyAI/minds14)), which means you'll need to resample the dataset to 16000Hz to use the pretrained Wav2Vec2 model:
- The Wav2Vec2 tokenizer is only trained on uppercase characters so you'll need to make sure the text matches the tokenizer's vocabulary:

In [ ]:
minds = minds.cast_column("audio", Audio(sampling_rate=16_000))
print(minds["train"].features['audio'])

def uppercase(example):
    return {"transcription": example["transcription"].upper()}

minds = minds.map(uppercase)

Now create a preprocessing function that:

1. Calls the `audio` column to load and resample the audio file.
2. Extracts the `input_values` from the audio file and tokenize the `transcription` column with the processor.

In [ ]:
def prepare_dataset(batch, processor):
    audio = batch["audio"]
    batch = processor(audio["array"],
                      sampling_rate=audio["sampling_rate"],
                      text=batch["transcription"])
    batch["input_length"] = len(batch["input_values"][0])
    return batch

processor = AutoProcessor.from_pretrained(model_name)
encoded_minds = minds.map(
    prepare_dataset,
    remove_columns=minds.column_names["train"],
    fn_kwargs={"tokenizer": tokenizer}
    num_proc=4
)

## Train

- HF Transformers don't have a data collator for ASR.
- We'll adapt the [DataCollatorWithPadding](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorWithPadding) to create a batch of examples.
- It'll also dynamically pad your text and labels to the length of the longest element in its batch (instead of the entire dataset) so they are a uniform length.
    - <font color="lightgreen">While it is possible to pad your text in the `tokenizer` function by setting `padding=True`, dynamic padding is more efficient.</font>

- Unlike other data collators, this specific data collator needs to apply a different padding method to `input_values` and `labels`:

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: AutoProcessor
    padding: Union[bool, str] = "longest"

    def __call__(
        self, features: list[dict[str, Union[list[int], torch.Tensor]]]
    ) -> dict[str, torch.Tensor]:
        # split inputs and labels since they have to be
        # of different lengths and need different padding methods
        input_features = [
            {"input_values": feature["input_values"][0]}
            for feature in features
        ]
        label_features = [
            {"input_ids": feature["labels"]} for feature in features
        ]
        # "input_ids" are ids per char in the transcription text
            # (processor.tokenizer.vocab_size = 32)
        # ??processor.pad:
            # the processor.feature_extractor.pad is applied on input_features
            # the processor.tokenizer.pad is applied on labels
        batch = self.processor.pad(input_features,
                                   padding=self.padding,
                                   return_tensors="pt")
        labels_batch = self.processor.pad(labels=label_features,
                                          padding=self.padding,
                                          return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

In [ ]:
def compute_metrics(pred, processor, evaluator, metric_name):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = np.where(
        pred.label_ids == -100,
        processor.tokenizer.pad_token_id,
        pred.label_ids
    )

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.tokenizer.batch_decode(
        label_ids,
        group_tokens=False # ??processor.tokenizer.convert_tokens_to_string
    )
    
    # print("debug --> inside compute_metrics")
    # print(f"pred_ids.shape: {pred_ids.shape}")
    # print(f"label_ids.shape: {label_ids.shape}")
    # print("pred sample:", pred_str[:3])
    # print("label sample:", label_str[:3])
    # print("empty preds:", sum(s.strip()=="" for s in pred_str), "/", len(pred_str))

    score = evaluator.compute(predictions=pred_str, references=label_str)
    return {metric_name: score}


model = AutoModelForCTC.from_pretrained(
    model_name,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)

metric_name = "wer"
wer = evaluate.load(metric_name)

In [ ]:
training_args = TrainingArguments(
    output_dir=ckpt_name,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    # num_train_epochs=num_train_epochs, # overridden by max_steps
    metric_for_best_model=metric_name,
    greater_is_better=False,
    push_to_hub=push_to_hub,
    max_steps=max_train_steps, # optimizer step
    warmup_steps=max_train_steps // 4,
    eval_steps=(max_train_steps // 8),
    save_steps=(max_train_steps // 8) * 2,
    logging_steps=max_train_steps // 16,
    
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    gradient_checkpointing=True,
    fp16=True,
    group_by_length=True, # reorders/buckets examples to reduce padding memory usage
    eval_strategy="steps",
    load_best_model_at_end=True,
    report_to = 'none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_minds["train"],
    eval_dataset=encoded_minds["test"],
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=partial(compute_metrics, metric_name=metric_name, evaluator=wer, processor=processor),
)

In [ ]:
torch.cuda.empty_cache()
trainer.train()
if push_to_hub: trainer.push_to_hub()

## Inference

### The simplest way

In [ ]:
audio_file = minds["train"][0]["audio"]['array']
model_name = f"{user_name}/{ckpt_name}"
transcriber = pipeline("automatic-speech-recognition", model=model_name)
print(transcriber(audio_file))

### More complex | More flexible

In [ ]:
model_name = f"{user_name}/{ckpt_name}"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCTC.from_pretrained(model_name)
sampling_rate = minds["train"][0]["audio"]["sampling_rate"]
inputs = processor(
    audio_file, sampling_rate=sampling_rate, return_tensors="pt"
)

with torch.no_grad():
    logits = model(**inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)
print(transcription)